# Análisis de sentimiento en Bluesky — Elecciones generales 29-N

**CEIABD · Versión local (VSCode / Jupyter)**

## Objetivo

Recoger publicaciones de Bluesky que mencionan a cada candidatura durante la campaña electoral, clasificar automáticamente su **sentimiento** (positivo, negativo o neutro) con un modelo de lenguaje en español, y comprobar a mano **cuánto acierta** ese modelo.

## Flujo de trabajo

```
 .env (credenciales)
      │
      ▼
 1. Comprobar entorno ─► 2. Cargar credenciales ─► 3. Iniciar sesión en Bluesky
                                                          │
      ┌───────────────────────────────────────────────────┘
      ▼
 4. Configurar búsqueda ─► 5. Recoger publicaciones ─► datos/bluesky_bruto.csv
                                   │
                                   ▼
 6. Limpiar y seudonimizar ─► 7. Clasificar sentimiento ─► datos/bluesky_clasificado.csv
                                   │
                                   ▼
 8–9. Resultados y gráficos ─► 10. Anotación manual ─► 11. Evaluación del modelo
```

## Librerías que se usan

| Librería | Para qué sirve en este cuaderno |
|---|---|
| `atproto` | Cliente oficial en Python del protocolo de Bluesky (AT Protocol). Permite iniciar sesión y buscar publicaciones. |
| `python-dotenv` | Lee el archivo `.env` y carga las credenciales como variables de entorno, para no escribirlas en el código. |
| `pandas` | Manejo de datos en tablas (*DataFrames*): limpiar, filtrar, agrupar y guardar en CSV. |
| `hashlib` *(incluida en Python)* | Calcula *hashes* SHA-256 para sustituir los nombres de usuario por códigos anónimos. |
| `pysentimiento` | Librería de análisis de opiniones en español. Descarga y ejecuta el modelo de sentimiento. |
| `transformers` + `torch` *(las instala `pysentimiento`)* | Motor que ejecuta el modelo de lenguaje (red neuronal tipo RoBERTa). |
| `matplotlib` | Dibuja los gráficos. |
| `scikit-learn` | Calcula las métricas de evaluación: exactitud, F1 y matriz de confusión. |
| `datetime`, `time`, `os`, `pathlib` *(incluidas en Python)* | Fechas, pausas entre peticiones, variables de entorno y rutas de archivos. |

## Antes de empezar

Sigue el `README.md` de la carpeta: crear el entorno virtual, instalar `requirements.txt`, rellenar el archivo `.env` y seleccionar el kernel en VSCode.

> **Normas:** no difundir resultados fuera del aula antes de las 20:00 del 29 de noviembre (art. 69.7 LOREG). Los resultados describen la conversación en una red social, **no** la intención de voto.

---
## 1. Comprobar el entorno

**Qué hace:** importa todas las librerías que vamos a usar y muestra sus versiones.

**Por qué:** si falta alguna, el error aparece aquí, al principio, y no a mitad del trabajo. Si ves `ModuleNotFoundError`, revisa que el kernel seleccionado en VSCode (arriba a la derecha) es el de la carpeta `.venv` y que has ejecutado `pip install -r requirements.txt`.

- `%matplotlib inline` hace que los gráficos aparezcan debajo de cada celda.
- `display()` muestra las tablas de pandas con formato (en Jupyter suele estar disponible, pero lo importamos para que funcione siempre).

In [ ]:
%matplotlib inline
import os, time, hashlib
import datetime as dt
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
from atproto import Client
import atproto, sklearn

print("pandas      ", pd.__version__)
print("atproto     ", atproto.__version__ if hasattr(atproto, "__version__") else "instalado")
print("scikit-learn", sklearn.__version__)
print("Entorno listo.")

---
## 2. Cargar las credenciales desde `.env`

**Qué hace:** lee el archivo `.env` (que está en la misma carpeta que este cuaderno) y recupera tres valores:

| Variable | Qué es |
|---|---|
| `BSKY_HANDLE` | Tu usuario de Bluesky, p. ej. `nombre.bsky.social` (sin `@`). |
| `BSKY_APP_PASSWORD` | Una **contraseña de aplicación** creada en Bluesky (Ajustes → Privacidad y seguridad → App Passwords). Nunca la contraseña principal. |
| `HASH_SAL` | Una cadena aleatoria secreta del grupo, usada para anonimizar usuarios (ver bloque 6). |

**Librerías:**
- `load_dotenv()` (de `python-dotenv`) copia el contenido de `.env` a las variables de entorno del proceso.
- `os.getenv("NOMBRE")` lee una variable de entorno; devuelve `None` si no existe.

**Por qué así:** las contraseñas nunca deben escribirse en el código. El `.gitignore` incluido impide que `.env` se suba a un repositorio por error.

También se crea la carpeta `datos/`, donde se guardarán los CSV.

In [ ]:
load_dotenv()

BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
HASH_SAL = os.getenv("HASH_SAL")

faltan = [n for n, v in {"BSKY_HANDLE": BSKY_HANDLE,
                         "BSKY_APP_PASSWORD": BSKY_APP_PASSWORD,
                         "HASH_SAL": HASH_SAL}.items() if not v]
if faltan:
    raise RuntimeError(f"Faltan en el archivo .env: {', '.join(faltan)}. Copia .env.example a .env y rellénalo.")

DATOS = Path("datos")
DATOS.mkdir(exist_ok=True)
print("Credenciales cargadas para", BSKY_HANDLE)

---
## 3. Iniciar sesión en Bluesky

**Qué hace:** crea un cliente de Bluesky y abre una sesión con tu usuario y tu contraseña de aplicación.

**Librería:** `atproto.Client` es el cliente oficial de Bluesky para Python.
- `client.login(usuario, contraseña)` se autentica contra el servidor de Bluesky y guarda el token de sesión dentro del objeto `client`.
- A partir de aquí, todas las peticiones que hagamos con `client` van autenticadas.

**Por qué iniciar sesión:** la búsqueda de publicaciones sin sesión puede devolver un error 403. Con sesión funciona de forma fiable.

**Errores habituales:** `Invalid identifier or password` → el usuario está mal escrito o has usado la contraseña principal en lugar de la de aplicación.

In [ ]:
client = Client()
client.login(BSKY_HANDLE, BSKY_APP_PASSWORD)
print("Sesión iniciada como", client.me.handle)

---
## 4. Configurar la búsqueda

**Qué hace:** define **qué** buscamos y **en qué fechas**.

- `ENTIDADES` es un diccionario: la **clave** es el nombre de la candidatura y el **valor** es la lista de términos que la identifican (siglas, nombre del candidato, hashtags).
- `INICIO` y `FIN` marcan el periodo. La campaña empezó el 13 de noviembre; `FIN` es hoy o el 29, lo que sea antes.
- `POR_TERMINO_Y_DIA` limita cuántas publicaciones se piden por cada término y cada día (máximo que permite la API en una petición: 100).
- `dias` es la lista de fechas, creada con `datetime.timedelta` para sumar días.

**Criterios para el diccionario:**
1. Incluir **todas** las candidaturas relevantes, con el mismo criterio para todas.
2. Añadir variantes **con y sin tilde** (mucha gente escribe sin tildes).
3. Evitar términos muy cortos o ambiguos (p. ej. «PP» solo, o «Vox» puede aparecer en otros contextos): traen ruido. Revisa una muestra tras la recogida.

**Coste:** el número de peticiones es `días × términos`. Con 11 días y 10 términos son 110 peticiones.

In [ ]:
ENTIDADES = {
    "PSOE": ["PSOE", "Pedro Sánchez", "Pedro Sanchez"],
    "PP":   ["Partido Popular", "Feijóo", "Feijoo"],
    "Vox":  ["Vox", "Abascal"],
    # Añadir el resto de candidaturas proclamadas...
}

INICIO = dt.date(2026, 11, 13)
FIN = min(dt.date.today(), dt.date(2026, 11, 29))
POR_TERMINO_Y_DIA = 100

dias = [INICIO + dt.timedelta(days=i) for i in range((FIN - INICIO).days + 1)]
n_terminos = sum(len(t) for t in ENTIDADES.values())
print(f"{len(dias)} días × {n_terminos} términos = {len(dias) * n_terminos} peticiones")

---
## 5. Recoger las publicaciones

**Qué hace:** para cada candidatura, cada término y cada día, pide a Bluesky las publicaciones en español que contienen ese término, y las guarda en una tabla.

**Cómo funciona la función `buscar()`:**
- Llama a `client.app.bsky.feed.search_posts(...)`, que corresponde al endpoint `app.bsky.feed.searchPosts` de la API de Bluesky.
- Parámetros que usamos:

| Parámetro | Significado |
|---|---|
| `q` | Texto a buscar. |
| `lang="es"` | Solo publicaciones marcadas como español. |
| `sort="latest"` | Las más recientes primero (alternativa: `"top"`, las más populares). |
| `limit` | Cuántos resultados (máx. 100). |
| `since` / `until` | Inicio y fin del intervalo, en formato ISO con zona horaria UTC (`Z`). |

- De cada publicación (`p`) guardamos: identificador único (`uri`), autor, fecha, texto, «me gusta» y republicaciones.

**Bucle principal:**
- `time.sleep(0.5)` hace una pausa de medio segundo entre peticiones para no superar los límites de uso de la API.
- `try/except` evita que un fallo puntual (red, límite de peticiones) detenga toda la recogida: se avisa y se continúa.

**Guardado:** el resultado se guarda en `datos/bluesky_bruto.csv`. Si el archivo ya existe y `VOLVER_A_RECOGER = False`, **se carga del disco** en lugar de volver a pedir todo a Bluesky. Así puedes reiniciar el cuaderno sin repetir la recogida.

**Tiempo estimado:** unos minutos, según el número de términos y días.

In [ ]:
VOLVER_A_RECOGER = False
ARCHIVO_BRUTO = DATOS / "bluesky_bruto.csv"

def buscar(q, dia, limite):
    since = f"{dia}T00:00:00Z"
    until = f"{dia + dt.timedelta(days=1)}T00:00:00Z"
    r = client.app.bsky.feed.search_posts(params={
        "q": q, "lang": "es", "sort": "latest", "limit": min(limite, 100),
        "since": since, "until": until})
    return [{"id": p.uri, "autor": p.author.handle, "fecha": p.record.created_at,
             "texto": p.record.text, "likes": p.like_count, "reposts": p.repost_count}
            for p in r.posts]

if ARCHIVO_BRUTO.exists() and not VOLVER_A_RECOGER:
    bruto = pd.read_csv(ARCHIVO_BRUTO)
    print(f"Cargado desde disco: {len(bruto)} publicaciones (pon VOLVER_A_RECOGER = True para repetir).")
else:
    filas = []
    for entidad, terminos in ENTIDADES.items():
        for t in terminos:
            for dia in dias:
                try:
                    for f in buscar(t, dia, POR_TERMINO_Y_DIA):
                        filas.append({**f, "entidad": entidad, "termino": t, "dia": str(dia)})
                except Exception as e:
                    print(f"Fallo en '{t}' {dia}: {type(e).__name__}")
                time.sleep(0.5)
            print(f"{entidad:10} / {t:20} → {len(filas)} acumuladas")
    bruto = pd.DataFrame(filas)
    bruto.to_csv(ARCHIVO_BRUTO, index=False, encoding="utf-8-sig")
    print("Total recogido:", len(bruto))

bruto.head()

---
## 6. Limpiar y seudonimizar

**Qué hace, paso a paso:**

1. **Textos vacíos:** `fillna("")` convierte los valores vacíos (`NaN`, que aparecen en publicaciones con solo una imagen) en texto vacío; después se descartan los textos de 3 caracteres o menos.
2. **Publicaciones con varias candidaturas:** una misma publicación puede aparecer al buscar «PSOE» y al buscar «Feijóo». En ese caso no sabríamos a quién se dirige el sentimiento, así que **se descarta**. Se detecta con `groupby("id")["entidad"].nunique()`: cuenta cuántas entidades distintas tiene cada publicación.
3. **Duplicados:** se eliminan las publicaciones repetidas (mismo `id`) y los textos idénticos (copias).
4. **Seudonimización:** el nombre de usuario se sustituye por un código.
   - `hashlib.sha256(...)` calcula una huella digital de 64 caracteres; nos quedamos con los 12 primeros.
   - Se le añade antes la **sal** secreta (`HASH_SAL`). Sin sal, cualquiera podría calcular el hash de un usuario conocido y encontrarlo en los datos.
   - El mismo usuario obtiene siempre el mismo código, así que se puede contar cuánto publica cada cuenta sin saber quién es.

**Librería:** `pandas` para todas las operaciones con la tabla; `hashlib` (incluida en Python) para el hash.

> Con la sal guardada, esto es **seudonimización** (RGPD): quien tenga la sal podría reidentificar. Al terminar la tarea, borrad la sal y `datos/bluesky_bruto.csv`.

In [ ]:
df = bruto.copy()

# 1. Textos vacíos o muy cortos
df["texto"] = df["texto"].fillna("").astype(str).str.strip()
df = df[df["texto"].str.len() > 3]

# 2. Descartar publicaciones asociadas a más de una candidatura
n_ent = df.groupby("id")["entidad"].nunique()
df = df[df["id"].isin(n_ent[n_ent == 1].index)]

# 3. Duplicados
df = df.drop_duplicates("id").drop_duplicates("texto").reset_index(drop=True)

# 4. Seudonimización de autores
df["autor"] = df["autor"].astype(str).map(
    lambda a: hashlib.sha256((HASH_SAL + a).encode()).hexdigest()[:12])

print(f"Publicaciones recogidas: {len(bruto)} → válidas tras la limpieza: {len(df)}")
print("\nPor candidatura:")
print(df["entidad"].value_counts())

---
## 7. Clasificar el sentimiento

**Qué hace:** pasa cada texto por un modelo de lenguaje que devuelve una de tres etiquetas:

| Etiqueta | Significado | Ejemplo |
|---|---|---|
| `POS` | Tono positivo | «Gran mitin hoy, ilusión a tope» |
| `NEG` | Tono negativo | «Otra mentira más, qué vergüenza» |
| `NEU` | Neutro o informativo | «El debate empieza a las 22:00» |

**Librerías:**
- `pysentimiento` ofrece la función `create_analyzer(task="sentiment", lang="es")`, que descarga y prepara el modelo **`pysentimiento/robertuito-sentiment-analysis`**.
- Ese modelo es **RoBERTuito**: una red neuronal tipo RoBERTa preentrenada con millones de tuits en español y ajustada para sentimiento con el corpus TASS 2020.
- Por debajo, `pysentimiento` usa `transformers` (Hugging Face) y `torch` (PyTorch) para ejecutar la red.
- Antes de clasificar, `pysentimiento` **preprocesa** cada texto: normaliza menciones (`@usuario`), enlaces, emojis y risas («jajaja»).

**`analizador.predict(lista)`** devuelve, para cada texto, un objeto con:
- `.output` → la etiqueta ganadora (`POS`, `NEG` o `NEU`).
- `.probas` → la probabilidad de cada etiqueta (las guardamos para poder ver cuándo el modelo duda).

**Primera ejecución:** descarga el modelo (unos 435 MB) y lo guarda en la caché de Hugging Face (`~/.cache/huggingface`). Las siguientes veces se carga del disco. El aviso sobre `HF_TOKEN` es informativo y se puede ignorar.

**Tiempo:** en un portátil sin GPU, varios minutos por cada mil textos.

In [ ]:
from pysentimiento import create_analyzer

analizador = create_analyzer(task="sentiment", lang="es")
pred = analizador.predict(df["texto"].tolist())

df["sent"] = [p.output for p in pred]
df["confianza"] = [max(p.probas.values()) for p in pred]

df.to_csv(DATOS / "bluesky_clasificado.csv", index=False, encoding="utf-8-sig")
print(df["sent"].value_counts())

Para comprobar a simple vista si el modelo tiene sentido, mira 10 textos al azar con su etiqueta y su confianza (0 a 1):

In [ ]:
pd.set_option("display.max_colwidth", 200)
display(df[["entidad", "texto", "sent", "confianza"]].sample(10, random_state=1))

---
## 8. Resultados por candidatura

**Qué hace:** calcula qué proporción de textos de cada candidatura es positiva, negativa o neutra, y un indicador resumen:

**Sentimiento neto** = proporción de positivos − proporción de negativos

Va de **−1** (todo negativo) a **+1** (todo positivo). Un valor de −0,3 significa que hay 30 puntos más de negativos que de positivos.

**Librerías:**
- `pd.crosstab(filas, columnas, normalize="index")` construye una tabla de frecuencias y la convierte en proporciones por fila (cada fila suma 1).
- `.plot(kind="barh", stacked=True)` (pandas usando `matplotlib`) dibuja barras horizontales apiladas.

**Cómo leerlo:** la columna `n` indica cuántos textos hay de cada candidatura. Con pocos textos (menos de 100), las proporciones son poco fiables.

In [ ]:
reparto = pd.crosstab(df["entidad"], df["sent"], normalize="index").round(3)
for c in ["NEG", "NEU", "POS"]:
    if c not in reparto:
        reparto[c] = 0.0
reparto["neto"] = reparto["POS"] - reparto["NEG"]
reparto["n"] = df["entidad"].value_counts()
display(reparto[["NEG", "NEU", "POS", "neto", "n"]].sort_values("neto"))

reparto[["NEG", "NEU", "POS"]].plot(kind="barh", stacked=True, figsize=(8, 3),
                                    color=["#d9534f", "#bbbbbb", "#5cb85c"])
plt.title("Reparto de sentimiento por candidatura")
plt.xlabel("Proporción de textos")
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

---
## 9. Evolución diaria durante la campaña

**Qué hace:** calcula el sentimiento neto de cada candidatura **cada día** y lo dibuja como una línea por candidatura.

**Cómo funciona:**
- `df["sent"].eq("POS")` crea una columna de verdadero/falso; la **media** de verdaderos es la proporción de positivos.
- `groupby(["dia", "entidad"])` agrupa por día y candidatura.
- `.unstack()` pasa las candidaturas a columnas, que es el formato que necesita `plot()` para dibujar una línea por candidatura.
- `plt.axhline(0)` dibuja una línea en el cero para distinguir días positivos y negativos.

**Qué buscar:** picos o caídas que coincidan con acontecimientos de campaña (debates, mítines, noticias). Ojo: un día con pocos textos puede dar picos que no significan nada; la tabla de abajo muestra cuántos hay.

In [ ]:
diario = (df.assign(pos=df["sent"].eq("POS"), neg=df["sent"].eq("NEG"))
            .groupby(["dia", "entidad"])[["pos", "neg"]].mean())
diario["neto"] = diario["pos"] - diario["neg"]

diario["neto"].unstack().plot(marker="o", figsize=(9, 4))
plt.axhline(0, color="grey", lw=0.8)
plt.title("Sentimiento neto diario por candidatura")
plt.ylabel("Neto (−1 a +1)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Número de textos por día y candidatura:")
display(df.groupby(["dia", "entidad"]).size().unstack(fill_value=0))

---
## 10. Anotación manual de 50 textos

**Para qué:** el modelo no es perfecto. Para saber cuánto acierta **en nuestros datos** (texto político, con ironía y jerga), necesitamos compararlo con etiquetas puestas por personas.

**Qué hace:**
- `df.sample(n=50, random_state=42)` elige 50 textos al azar. `random_state` fija la semilla para que la muestra sea siempre la misma.
- Te muestra cada texto **sin** la predicción del modelo (para no influirte) y te pide una etiqueta con `input()`.

**En VSCode**, el cuadro para escribir aparece **en la parte superior de la ventana** (no debajo de la celda). Escribe y pulsa Intro:

| Tecla | Etiqueta |
|---|---|
| `p` | Positivo |
| `n` | Negativo |
| `u` | Neutro |
| `s` | Saltar (no se entiende o no trata de política) |

**Criterio:** etiqueta el tono del texto hacia la candidatura mencionada. Ante la duda entre neutro y otra opción, elige neutro.

Las respuestas se guardan en `datos/anotacion_manual.csv`, así no se pierden si cierras el cuaderno.

In [ ]:
ARCHIVO_ANOT = DATOS / "anotacion_manual.csv"
mapa = {"p": "POS", "n": "NEG", "u": "NEU"}

muestra = df.sample(n=min(50, len(df)), random_state=42).reset_index(drop=True)
anot = []
for i, fila in muestra.iterrows():
    print(f"\n[{i+1}/{len(muestra)}] ({fila['entidad']})\n{fila['texto']}")
    r = ""
    while r not in ("p", "n", "u", "s"):
        r = input(f"[{i+1}/{len(muestra)}] p / n / u / s > ").strip().lower()
    anot.append(mapa.get(r))

muestra["humano"] = anot
muestra.to_csv(ARCHIVO_ANOT, index=False, encoding="utf-8-sig")
print("\nAnotación guardada en", ARCHIVO_ANOT)

---
## 11. Evaluar el modelo

**Qué hace:** compara tus etiquetas (`humano`) con las del modelo (`sent`) usando métricas de `scikit-learn`:

| Métrica | Qué mide |
|---|---|
| **Exactitud** (`accuracy_score`) | Porcentaje de textos en que el modelo coincide contigo. Puede engañar si una clase domina (p. ej. 80 % negativos). |
| **F1 macro** (`f1_score(..., average="macro")`) | Media del F1 de cada clase, dando el mismo peso a las tres. Mejor indicador cuando las clases están desequilibradas. |
| **Informe por clase** (`classification_report`) | Para cada etiqueta: *precision* (de lo que el modelo dijo POS, cuánto lo era), *recall* (de lo que era POS, cuánto encontró) y F1. |
| **Matriz de confusión** (`ConfusionMatrixDisplay`) | Tabla: filas = tu etiqueta, columnas = la del modelo. La diagonal son aciertos; fuera de ella, qué confunde con qué. |

Al final se listan los **errores** para comentarlos en el informe. Busca patrones: ironía, negaciones, textos que critican a un partido citando a otro, etc.

Si ya anotaste antes y has reiniciado el cuaderno, esta celda carga la anotación desde `datos/anotacion_manual.csv`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

if "muestra" not in globals() or "humano" not in muestra:
    muestra = pd.read_csv(ARCHIVO_ANOT)

ev = muestra.dropna(subset=["humano"])
print(f"Textos evaluados: {len(ev)}")
print("Exactitud:", round(accuracy_score(ev["humano"], ev["sent"]), 3))
print("F1 macro: ", round(f1_score(ev["humano"], ev["sent"], average="macro"), 3))
print()
print(classification_report(ev["humano"], ev["sent"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(ev["humano"], ev["sent"], labels=["NEG", "NEU", "POS"])
plt.title("Humano (filas) frente a modelo (columnas)")
plt.show()

print("Errores del modelo:")
display(ev.loc[ev["humano"] != ev["sent"], ["entidad", "texto", "humano", "sent", "confianza"]])

---
## 12. Preguntas para el informe

1. ¿Qué candidatura tiene el sentimiento neto más alto y cuál el más bajo? ¿Cambia a lo largo de la campaña? ¿Coincide con algún acontecimiento (debate, mitin, noticia)?
2. ¿En qué tipo de textos se equivoca más el modelo? Pon ejemplos (ironía, negaciones, críticas a un partido citando a otro…).
3. ¿Los errores tienen más confianza baja que los aciertos? ¿Qué pasaría si descartáramos las predicciones con confianza < 0,6?
4. ¿Quién usa Bluesky en España? ¿Por qué esta muestra no sirve para predecir el resultado electoral?
5. Tras el 29-N: compara tu tabla del bloque 8 con el escrutinio oficial del Ministerio del Interior. ¿Qué se parece y qué no? ¿Por qué?

## Al terminar

- Borrad la carpeta `datos/` con los datos en bruto y la línea `HASH_SAL` del `.env`.
- Borrad la contraseña de aplicación en Bluesky (Ajustes → Privacidad y seguridad → App Passwords).